Written by: Trisha Jha

Last update: Jul, 2026


# CroP-DYNO Tutorial

## Overview
CroP-DYNO (cross-population prioritized dynamical nonlinear interaction model) is a method for nonlinear dynamical modeling of interactions between two simultaneously recorded neural populations. Given the activity of a source population Y and a target population Z, CroP-DYNO learns a dynamical model of population Y while prioritizing the dynamics that are shared with population Z. For more details, see our manuscript: Jha T, Song D and Shanechi M M 2026 Prioritized nonlinear
modeling of shared dynamics across neural populations *J. Neural Eng.*
https://doi.org/10.1088/1741-2552/ae9bee

CroP-DYNO's implementation is derived from the [DPAD](https://github.com/ShanechiLab/DPAD) codebase (Sani, Pesaran, and Shanechi, *Nature Neuroscience*, 2024).

## Model
Given the activity of a source population, $y_k$, and a target population, $z_k$, CroP-DYNO learns a latent state $x_k$, composed of two components: $x_k^{(1)}$, the dynamics shared between populations Y and Z, and $x_k^{(2)}$, the within-population Y dynamics not shared with population Z. The model is formulated as:

$$ x_{k+1} = A'(x_k) + K(y_k) $$

$$ y_k = C_y(x_k) + e_k $$

$$ z_k = C_z(x_k) + \epsilon_k $$

Multi-input multi-output functions $A'$ (latent state dynamics), $K$ (mapping from source population Y activity to latent states), $C_y$ (source population Y readout), and $C_z$ (target population Z readout) define the model. CroP-DYNO allows the user to flexibly set any of these four mappings to be linear or nonlinear, which in turn enables testing hypotheses about where nonlinearity in the cross-population interaction originates.

*Note: in the CroP-DYNO paper these two populations are referred to as population A and population B; here we call them population Y and population Z to match the naming used in the code below.*


# Using CroP-DYNO
## Modeling data
To use CroP-DYNO, import the model class:
```python
from CroPDYNO.CroPDYNOModel import CroPDYNOModel
```

You can then use it to model cross-population data as:
```python
idSys = CroPDYNOModel()
methodCode = 'CroPDYNO_Cz1HL64U'  # Only Cz (the target-population Z readout) will be nonlinear, 1 Hidden layer with 64 Units.
args = CroPDYNOModel.prepare_args(methodCode)
idSys.fit(Y=yTrain, Z=zTrain, nx=6, n1=2, **args)
```

With the following arguments:
- `Y`: Source population Y activity. Dimensions are: data x time.
- `Z`: Target population Z activity, i.e. the cross-population prediction objective. Dimensions are: data x time.
- `nx`: the total dimension of the latent state in the model.
- `n1`: the number of latent state dimensions dedicated to shared cross-population dynamics ($x_k^{(1)}$).
- `args`: other arguments specifying which mappings are linear vs. nonlinear.

`CroPDYNOModel.prepare_args` takes a `methodCode` string and returns the arguments to pass to `idSys.fit`. In the example above, `methodCode` is `'CroPDYNO_Cz1HL64U'`, which makes only $C_z$ nonlinear while $A'$, $K$, and $C_y$ remain linear. You can inspect the output of `CroPDYNOModel.prepare_args` to see how the architecture of each mapping is specified.

After running the above, the learned model is stored in the object `idSys`.

## Using the model for cross-population prediction and self-prediction
For a learned model `idSys`, use `predict` to extract the latent state and predict both populations' activity given new source population Y activity:
```python
zPred, yPred, xPred = idSys.predict(yTest)
```
With the argument:
- `yTest`: Source population Y activity in the test data. Dimensions are: time x data dimension.

And with outputs (all dimensions are time x data dimension):
- `zPred`: Prediction of target population Z activity from past source population Y activity (i.e., cross-population prediction).
- `yPred`: Self-prediction of source population Y activity from its own past.
- `xPred`: The latent state extracted at each time point.

Note that inference only requires source population Y activity, consistent with CroP-DYNO's design for settings where only one population is accessible at inference time.

We next go through a complete worked example on simulated data.


# Example: CroP-DYNO on simulated cross-population data
In this example, we use CroP-DYNO to model simulated data from a system with a known ground-truth model, and evaluate cross-population prediction performance against that ground truth. We use a simulated dataset where only $C_z$ is nonlinear. Both populations are multi-channel (3 channels each).

First, we import CroP-DYNO and a few other useful tools.


In [11]:
# CroP-DYNO requires Python 3.11 (see the repository README for setup).
# This prints your Python version as a quick check.
!python --version


Python 3.11.15


In [12]:
import logging
import os
import sys

sys.path.insert(0, os.path.join("..", ".."))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("")), "..", ".."))

# Setting up logs
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

import coloredlogs

coloredlogs.install(
    level=logging.INFO,
    fmt="%(asctime)s %(name)s [%(filename)s > %(lineno)s] %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    logger=logger,
)

# Prepare plots
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["figure.facecolor"] = "w"
matplotlib.rcParams["axes.facecolor"] = "w"
matplotlib.rcParams.update(
    {"font.size": 16, "axes.titlesize": 14, "axes.labelsize": 14}
)

# Import CroP-DYNO and supporting tools
import CroPDYNO
from CroPDYNO.CroPDYNOModel import CroPDYNOModel
from CroPDYNO.tools.evaluation import evalPrediction
from CroPDYNO.tools.file_tools import pickle_load

# Set random seeds for exact reproducibility
seed = 42
import random

import tensorflow as tf

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)


Let's load data from an example ground-truth model where $C_z$ is nonlinear. We treat the first signal as source population Y activity ($y_k$) and the second signal as target population Z activity ($z_k$):


In [13]:
# Load data

data_dir = os.path.join(os.path.dirname(CroPDYNO.__file__), "..", "..", "data")
fName = "sample_data.p"

sample_data_path = os.path.join(data_dir, fName)

print("Loading example data from {}".format(sample_data_path))
data = pickle_load(sample_data_path)
# This data is from a ground-truth model with a nonlinear Cz.

maxN = int(2 * 1e4)  # Number of time points to use in this notebook
N = np.min((maxN, data["y"].shape[0]))
y = data["y"][:N, :]  # Source population Y activity
z = data["z"][:N, :]  # Target population Z activity
trueModel = data["csys"]  # The object containing the true simulated model

n1 = len(
    trueModel.zDims
)  # The number of latent state dimensions shared with population Z (cross-population states)

nx = n1  # Only running stage 1 (extract cross-population states); set nx > n1 to also learn residual within-population Y dynamics

allYData, allZData = y, z


Loading example data from /Users/trishajani/Documents/PhD/Research/Projects/CroPDYNO/source/CroPDYNO/example/../../CroPDYNO/../../data/sample_data.p


Let's separate the data into training and test segments.


In [14]:
# Separate data into training and test data:
trainInds = np.arange(np.round(0.9 * allYData.shape[0]), dtype=int)
testInds = np.arange(1 + trainInds[-1], allYData.shape[0])
yTrain = allYData[trainInds, :]
yTest = allYData[testInds, :]
zTrain = allZData[trainInds, :]
zTest = allZData[testInds, :]


## Section 1: Ideal vs. nonlinear $C_z$ vs. linear CroP-DYNO

Training the model. We fit CroP-DYNO with the ground-truth latent dimensions, using a nonlinear readout for population Z ($C_z$) and keeping the other three mappings linear:


In [15]:
# Fit CroP-DYNO to the training data:
idSys = CroPDYNOModel()

methodCode = "CroPDYNO_Cz1HL64U_ErSV16_skipCy" 
#ErSV16: changes early stopping to 16 epochs, based on an inner validation set (last 20% of the training data). Without this, by default early stopping is 3 epochs and based on training loss.
#skipCy: skip fitting Cy parameter as it is not relevant to cross-region prediction (save time for this tutorial)

args = CroPDYNOModel.prepare_args(methodCode)

epochs = 2000

idSys.fit(Y=yTrain.T, Z=zTrain.T, nx=nx, n1=n1, epochs=epochs, **args)


2026-07-13 15:27:39 DPAD.DPADModel [DPADModel.py > 907] INFO Early stopping measure is "val_loss". Includes validation loss, so we will ignore any provided validation data and creating validation data fom the training data to avoid overfitting
2026-07-13 15:27:39 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 15:27:39 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 15:27:39 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 15:27:39 absl [optimizer.py > 70] WARNIN

Inference on test data and computing cross-population prediction performance. Inference uses only source population Y activity, consistent with CroP-DYNO's causal inference from the source population:


In [16]:
# Inference:
print("Computing predictions for the learned model")
zTestPred, yTestPred, xTestPred = idSys.predict(yTest)

# Compute CC of cross-population prediction (z from y)
zCC = evalPrediction(zTest, zTestPred, "CC")

# For comparison, do the same using the true model
print("Computing predictions for the true model (slower due to the analytical symbolic implementation of nonlinearity)")
zTestPredIdeal, yTestPredIdeal, xTestPredIdeal = trueModel.predict(yTest)
zCCIdeal = evalPrediction(zTest, zTestPredIdeal, "CC")


print(
    f"Cross-population prediction (Z from Y) CC:\n"
    f"  CroP-DYNO (Cz nonlinear) => {np.mean(zCC):.3g}\n"
    f"  Ideal using true model => {np.mean(zCCIdeal):.3g}"
)

Computing predictions for the learned model
Computing predictions for the true model (slower due to the analytical symbolic implementation of nonlinearity)


applying Cz: 100%|██████████| 2000/2000 [00:01<00:00, 1195.30it/s]

Cross-population prediction (Z from Y) CC:
  CroP-DYNO (Cz nonlinear) => 0.705
  Ideal using true model => 0.764


## Comparison to a fully linear CroP-DYNO model

As a baseline, we also fit a fully linear CroP-DYNO model (all  mappings linear) to see how much the nonlinear $C_z$ actually helps cross-population prediction.


In [17]:
# Fit a fully linear CroP-DYNO model
methodCode = "CroPDYNO_ErSV16_skipCy"  # No nonlinearity terms; ErSV16 (early stopping) and skipCy still apply

idSysLin = CroPDYNOModel()
argsLin = CroPDYNOModel.prepare_args(methodCode)
idSysLin.fit(Y=yTrain.T, Z=zTrain.T, nx=nx, n1=n1, epochs=epochs, **argsLin)

# Inference and cross-population prediction CC
print("Computing predictions for the linear model")
zTestPredLin, yTestPredLin, xTestPredLin = idSysLin.predict(yTest)
zCCLin = evalPrediction(zTest, zTestPredLin, "CC")



print(
    f"Cross-population prediction (Z from Y) CC:\n"
    f"  Linear CroP-DYNO => {np.mean(zCCLin):.3g}\n"
    f"  CroP-DYNO (Cz nonlinear) => {np.mean(zCC):.3g}"
)

2026-07-13 15:37:48 DPAD.DPADModel [DPADModel.py > 907] INFO Early stopping measure is "val_loss". Includes validation loss, so we will ignore any provided validation data and creating validation data fom the training data to avoid overfitting
2026-07-13 15:37:48 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 15:37:48 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 15:37:48 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 15:37:48 absl [optimizer.py > 70] WARNIN

Computing predictions for the linear model
Cross-population prediction (Z from Y) CC:
  Linear CroP-DYNO => 0.198
  CroP-DYNO (Cz nonlinear) => 0.705


## Section 2: Flexible CroP-DYNO — automatically searching for the best nonlinearity

Above, we manually chose `'CroPDYNO_Cz1HL64U_ErSV16_skipCy'`, i.e., we assumed only $C_z$ should be nonlinear. Instead of guessing, flexible CroP-DYNO can search over which of the three mappings relevant to cross-population prediction ($A'$, $K$, $C_z$) should be linear vs. nonlinear. It does so by using an inner cross-validation on the training data, and picks the configuration that best predicts population Z. This runs in two steps: (1) search for the best nonlinearity configuration, then (2) fit one final model on all training data using that configuration.


In [18]:
# Step 1: search for the best nonlinearity configuration via inner cross-validation
from CroPDYNO.tools.flexible import fitCroPDYNOWithFlexibleNonlinearity

settings = {}
settings["min_cores_to_enable_parallelization"] = 100  # If fewer cores than this are available, runs will not be parallelized

settings["fit_epochs"] = 2000  # Same max-epochs cap as above; fitting stops early upon convergence

# Search space: A', K, and Cz each tried as linear (0 hidden layers) or nonlinear (1 hidden layer, 64 units) => 2^3 = 8 cases
# (Cy is not searched here and is skipped as it is not relevant for cross-region prediction),
# selected via 2-fold inner cross-validation (iCVF2) on cross-population (z) prediction
methodCode = "CroPDYNO_GSUT_iCVF2_AKCz1HL64U_ErSV16_skipCy"


saveDir = os.path.join("results")  # Where the inner cross-validation results are saved

# Typical runtime: sum (no parallelization) or max (full parallelization) of run time across the 8 candidate configurations
selectedMethodCode, iCVRes = fitCroPDYNOWithFlexibleNonlinearity(
    yTrain, Z=zTrain, nx=nx, n1=n1, settings=settings, methodCode=methodCode, saveDir=saveDir
)

print(f"\n\nSelected nonlinearity setting: {selectedMethodCode}")


2026-07-13 15:41:03 DPAD.tools.flexible [flexible.py > 249] INFO Running innerCV to choose among the following 8 variants:
(1) DPAD_ErSV16_skipCy
(2) DPAD_A1HL64U_ErSV16_skipCy
(3) DPAD_K1HL64U_ErSV16_skipCy
(4) DPAD_Cz1HL64U_ErSV16_skipCy
(5) DPAD_AK1HL64U_ErSV16_skipCy
(6) DPAD_ACz1HL64U_ErSV16_skipCy
(7) DPAD_KCz1HL64U_ErSV16_skipCy
(8) DPAD_AKCz1HL64U_ErSV16_skipCy
2026-07-13 15:41:03 DPAD.tools.flexible [flexible.py > 637] INFO Assuming new run because no result file was found in results/f1637131_2dacdb37_N/DPAD_GSUT_iCVF2_AKCz1HL64U_ErSV16_skipCy/.p
2026-07-13 15:41:03 DPAD.tools.flexible [flexible.py > 1381] INFO Fold 1/2: training samples: 9000, test samples=9000
2026-07-13 15:41:03 DPAD.tools.flexible [flexible.py > 1754] INFO Running fold 1 (NTrain=9000, NTest=9000)
2026-07-13 15:41:03 DPAD.tools.flexible [flexible.py > 1849] INFO Signal dimensions: y => (18000, 3), z => (18000, 3), u => None
2026-07-13 15:41:03 DPAD.tools.flexible [flexible.py > 1856] INFO Method codes to fi

Considering 8 possible combinations of location/type for nonlinearities (this might take a while)...


2026-07-13 15:41:03 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 15:41:03 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 15:41:03 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 15:41:03 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 15:41:03 absl [optimizer.py > 70] WARNING At this



Selected nonlinearity setting: CroPDYNO_Cz1HL64U_ErSV16_skipCy


The flexible search found that $C_z$-only is the best nonlinearity configuration. This means the flexible search works as intended; it recovered the correct configuation. Next, we can fit a final model on all the training data using this selected configuration.

In [19]:
# Step 2: fit one final model on all training data using the selected nonlinearity configuration
idSysFlex = CroPDYNOModel()
argsFlex = CroPDYNOModel.prepare_args(selectedMethodCode)
idSysFlex.fit(Y=yTrain.T, Z=zTrain.T, nx=nx, n1=n1, epochs=epochs, **argsFlex)

# Inference and cross-population prediction CC
print("Computing predictions for the flexible model")
zTestPredFlex, yTestPredFlex, xTestPredFlex = idSysFlex.predict(yTest)
zCCFlex = evalPrediction(zTest, zTestPredFlex, "CC")

print(
    f"Cross-population prediction (Z from Y) CC:\n  Flexible CroP-DYNO ({selectedMethodCode}) => {np.mean(zCCFlex):.3g}"
)



2026-07-13 16:14:37 DPAD.DPADModel [DPADModel.py > 907] INFO Early stopping measure is "val_loss". Includes validation loss, so we will ignore any provided validation data and creating validation data fom the training data to avoid overfitting
2026-07-13 16:14:38 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 16:14:38 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 16:14:38 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 16:14:38 absl [optimizer.py > 70] WARNIN

Computing predictions for the flexible model
Cross-population prediction (Z from Y) CC:
  Flexible CroP-DYNO (CroPDYNO_Cz1HL64U_ErSV16_skipCy) => 0.703


## Section 3: Causal vs. noncausal (smoothing) CroP-DYNO

So far, every model has been causal in time: prediction of $z_k$ only uses past activity of population Y. CroP-DYNO can also run noncausally (smoothing), where the RNN is bidirectional and estimates $z_k$ using both past *and future* activity of population Y. This is no longer causal in time, as it requires the full Y activity, but it can improve prediction accuracy.

Adding `_xSmth` to the `methodCode` string enables this bidirectional/noncausal mode. To isolate the effect of causal vs. noncausal on its own, we keep everything else the same as our original ground-truth-nonlinearity model (only $C_z$ nonlinear) and compare below.


In [20]:
# Fit the noncausal (smoothing) counterpart of our original nonlinear-Cz model:
methodCode = "CroPDYNO_Cz1HL64U_ErSV16_skipCy_xSmth"  # now noncausal (smoothing)

idSysSmth = CroPDYNOModel()
argsSmth = CroPDYNOModel.prepare_args(methodCode)
idSysSmth.fit(Y=yTrain.T, Z=zTrain.T, nx=nx, n1=n1, epochs=epochs, **argsSmth)


2026-07-13 16:15:19 DPAD.DPADModel [DPADModel.py > 907] INFO Early stopping measure is "val_loss". Includes validation loss, so we will ignore any provided validation data and creating validation data fom the training data to avoid overfitting
2026-07-13 16:15:19 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 16:15:19 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 16:15:19 absl [optimizer.py > 70] WARNING At this time, the v2.11+ optimizer `tf.keras.optimizers.Adam` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.Adam`.
2026-07-13 16:15:19 absl [optimizer.py > 70] WARNIN

In [21]:
# Inference and cross-population prediction CC
print("Computing predictions for the noncausal (smoothing) model")
zTestPredSmth, yTestPredSmth, xTestPredSmth = idSysSmth.predict(yTest)
zCCSmth = evalPrediction(zTest, zTestPredSmth, "CC")

print(
    f"Cross-population prediction (Z from Y) CC:\n"
    f"  CroP-DYNO (Cz nonlinear, causal) => {np.mean(zCC):.3g}\n"
    f"  CroP-DYNO (Cz nonlinear, noncausal/smoothing) => {np.mean(zCCSmth):.3g}"
)


Computing predictions for the noncausal (smoothing) model
Cross-population prediction (Z from Y) CC:
  CroP-DYNO (Cz nonlinear, causal) => 0.705
  CroP-DYNO (Cz nonlinear, noncausal/smoothing) => 0.893
